In [ ]:
                                             # Part 1

In [ ]:
# Task 1.1

In [ ]:
import csv
import json

In [ ]:
with open("appointments.csv", newline="") as file:

    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    rows = list(reader)

    print("Total rows:", len(rows))

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [ ]:
def parse_row(row):

    row = row.copy()

    row["age"] = int(row["age"])

    row["fee_inr"] = int(row["fee_inr"])

    row["prior_no_shows"] = int(row["prior_no_shows"])

    row["distance_km"] = float(row["distance_km"])

    return row

In [ ]:
accepted = []
rejected = []

for row in rows:

    try:

        converted_row = parse_row(row)

        accepted.append(converted_row)

    except ValueError:

        rejected.append(row)

print("Accepted rows:", len(accepted))

print("Rejected rows:", len(rejected))

Accepted rows: 988
Rejected rows: 32


In [ ]:
with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected records saved successfully!")

Rejected records saved successfully!


In [ ]:
# Task 1.2

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("appointments.csv")

print("Shape:", df.shape)

print("\nDataFrame Information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nNumber of exact duplicate rows:")
print(df.duplicated().sum())

Shape: (1020, 14)

DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values in each column:
appointment_id    

In [ ]:
# Removing exact duplicate rows
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [ ]:
# Cleaning the booking_channel column

print("Unique values before cleaning:")
print(df["booking_channel"].unique())

df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("\nUnique values after cleaning:")
print(df["booking_channel"].unique())

print("\nNumber of unique booking channels:", df["booking_channel"].nunique())

Unique values before cleaning:
['Phone' 'Walk-in' 'App' ' walk-in' 'APP ' 'Walk-In' 'Phone ' 'PHONE'
 'app' ' App' 'walk-in' 'phone' 'Walk-in ' ' phone' 'WALK-IN' 'App ']

Unique values after cleaning:
['Phone' 'Walk-In' 'App']

Number of unique booking channels: 3


In [ ]:
# Converting the date columns

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")

df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Date columns converted successfully.")

print(df[["booked_on", "appt_date"]].dtypes)

Date columns converted successfully.
booked_on    datetime64[ns]
appt_date    datetime64[ns]
dtype: object


In [ ]:
# Removing rows with missing appointment dates or status

df = df.dropna(subset=["appt_date", "status"])

print("Final rows after removing missing appt_date and status:", len(df))

print("\nFinal shape:", df.shape)

Final rows after removing missing appt_date and status: 990

Final shape: (990, 14)


In [ ]:
# Task 1.3

In [ ]:
# Replacing ages below 1 or above 100 with missing values
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = pd.NA

print("Missing ages after handling invalid values:", df["age"].isnull().sum())

Missing ages after handling invalid values: 17


In [ ]:
# Calculating median
age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Median age:", age_median)
print("Median distance:", distance_median)

# Filling missing values with median
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

# Verifying
print("\nMissing ages after filling:", df["age"].isnull().sum())
print("Missing distances after filling:", df["distance_km"].isnull().sum())

Median age: 48.0
Median distance: 8.25

Missing ages after filling: 0
Missing distances after filling: 0


In [ ]:
import numpy as np

In [ ]:
# Distance outlier limits
Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

print("Distance Q1:", Q1_distance)
print("Distance Q3:", Q3_distance)
print("Distance IQR:", IQR_distance)
print("Lower fence:", lower_distance)
print("Upper fence:", upper_distance)

Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625


In [ ]:
# Counting values outside the fences before capping
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()

# Capping outliers
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

print("Distance values capped:", distance_capped)

Distance values capped: 26


In [ ]:
# Fee outlier limits
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)
print("Lower fence:", lower_fee)
print("Upper fence:", upper_fee)

# Counting values outside the fences
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

# Cappping outliers
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

print("Fee values capped:", fee_capped)

Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Fee values capped: 3


In [ ]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Maximum fee after capping:", df["fee_inr"].max())
print("Rows remaining:", len(df))

Maximum fee after capping: 1219
Rows remaining: 990


In [ ]:
#  Task 1.4

In [ ]:
# Calculating the number of days between booking and appointment
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Converting appointment status into a binary target
df["no_show"] = (df["status"] == "No-Show").astype(int)


# Checking the new columns
print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())

   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1


In [ ]:
# Converting selected columns into a NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# Calculating the mean of each column
column_means = np.mean(X, axis=0)

print("Column means:", column_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [ ]:
# Creating a Boolean mask
long_lead_mask = df["lead_days"] > 14

# Counting appointments with lead_days greater than 14
long_lead_count = long_lead_mask.sum()

# Calculating their percentage
long_lead_percentage = long_lead_mask.mean() * 100

print("Appointments with lead_days > 14:", long_lead_count)
print("Percentage:", round(long_lead_percentage, 1), "%")

Appointments with lead_days > 14: 296
Percentage: 29.9 %


In [ ]:
# Extracting distance values as a NumPy array
distance = df["distance_km"].to_numpy()

# Calculating z-scores
z_scores = (distance - np.mean(distance)) / np.std(distance)

# Counting values where absolute z-score is greater than 2
distance_outliers = np.abs(z_scores) > 2

print("Number of distance outliers (|z| > 2):", distance_outliers.sum())

Number of distance outliers (|z| > 2): 53


In [ ]:
# Task 1.5

In [ ]:
clinics = pd.read_csv("clinics.csv")

print("Shape:", clinics.shape)
print(clinics.head())

Shape: (8, 6)
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873


In [ ]:
import requests

In [ ]:
temperatures = []

for index, clinic in clinics.iterrows():

    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None

    for attempt in range(2):

        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code == 200:

                data = response.json()

                hourly = data.get("hourly", {})
                temperature_data = hourly.get("temperature_2m", [])

                if temperature_data:
                    temp_c = temperature_data[0]
                    break

                else:
                    print(f"Attempt {attempt + 1}: Hourly temperature missing for {clinic['clinic_name']}")

            else:
                print(f"Attempt {attempt + 1}: HTTP {response.status_code} for {clinic['clinic_name']}")

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: Request failed for {clinic['clinic_name']}: {e}")

    if temp_c is None:
        print(f"Failed to fetch temperature for {clinic['clinic_name']} after 2 attempts.")

    temperatures.append(temp_c)

clinics["temp_c"] = temperatures

print("\nTemperature fetching completed.")
print(clinics[["clinic_name", "city", "temp_c"]])


Temperature fetching completed.
                clinic_name     city  temp_c
0     Kothrud Family Clinic     Pune    18.3
1      Hadapsar Care Centre     Pune    20.0
2    Sitabuldi Health Point   Nagpur    20.4
3      Malviya Nagar Clinic   Jaipur    16.2
4     Vaishali Nagar Clinic   Jaipur    16.2
5  Edappally Medical Centre    Kochi    26.2
6        Vijay Nagar Clinic   Indore    16.6
7    Gomti Nagar Care Point  Lucknow    16.9


In [ ]:
clinics.to_csv("clinics_enriched.csv", index=False)

print("clinics_enriched.csv saved successfully!")

clinics_enriched.csv saved successfully!


In [ ]:
print("Shape:", clinics.shape)

print("\nMissing temperatures:")
print(clinics["temp_c"].isnull().sum())

print("\nData types:")
print(clinics.dtypes)

print("\nFinal dataset:")
print(clinics)

Shape: (8, 7)

Missing temperatures:
0

Data types:
clinic_id       object
clinic_name     object
city            object
tier            object
latitude       float64
longitude      float64
temp_c         float64
dtype: object

Final dataset:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    18.3  
1    20.0  
2    20.4

In [ ]:
# Task 1.6

In [ ]:
# Sorting appointments by appointment_id
df = df.sort_values("appointment_id")

# Converting dates into YYYY-MM-DD format
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# Selecting columns in the required order
final_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

appointments_clean = df[final_columns].copy()

print("Final shape:", appointments_clean.shape)
print("\nColumn names:")
print(appointments_clean.columns.tolist())

print("\nFirst 5 rows:")
print(appointments_clean.head())

Final shape: (990, 16)

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

First 5 rows:
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
4          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
5          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation 

In [ ]:
# Checking duplicate appointment IDs
print("Duplicate appointment IDs:",
      appointments_clean["appointment_id"].duplicated().sum())

# Checking missing values
print("\nMissing values in each column:")
print(appointments_clean.isnull().sum())

# Checking total missing values
print("\nTotal missing values:",
      appointments_clean.isnull().sum().sum())

Duplicate appointment IDs: 0

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0


In [ ]:
appointments_clean.to_csv("appointments_clean.csv", index=False)

print("appointments_clean.csv exported successfully!")

appointments_clean.csv exported successfully!


In [ ]:
def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"

    return str(value)


with open("load_data.sql", "w") as file:

    # Insert clinics first
    for _, clinic in clinics.iterrows():

        sql = f"""
INSERT INTO clinics
(clinic_id, clinic_name, city, tier, latitude, longitude, temp_c)
VALUES (
    {sql_value(clinic['clinic_id'])},
    {sql_value(clinic['clinic_name'])},
    {sql_value(clinic['city'])},
    {sql_value(clinic['tier'])},
    {sql_value(clinic['latitude'])},
    {sql_value(clinic['longitude'])},
    {sql_value(clinic['temp_c'])}
);
"""

        file.write(sql)

    # Insert appointments
    for _, appointment in appointments_clean.iterrows():

        sql = f"""
INSERT INTO appointments
(appointment_id, patient_id, clinic_id, booked_on, appt_date,
 age, gender, distance_km, fee_inr, booking_channel,
 visit_type, reminder_sent, prior_no_shows, status,
 lead_days, no_show)
VALUES (
    {sql_value(appointment['appointment_id'])},
    {sql_value(appointment['patient_id'])},
    {sql_value(appointment['clinic_id'])},
    {sql_value(appointment['booked_on'])},
    {sql_value(appointment['appt_date'])},
    {sql_value(appointment['age'])},
    {sql_value(appointment['gender'])},
    {sql_value(appointment['distance_km'])},
    {sql_value(appointment['fee_inr'])},
    {sql_value(appointment['booking_channel'])},
    {sql_value(appointment['visit_type'])},
    {sql_value(appointment['reminder_sent'])},
    {sql_value(appointment['prior_no_shows'])},
    {sql_value(appointment['status'])},
    {sql_value(appointment['lead_days'])},
    {sql_value(appointment['no_show'])}
);
"""

        file.write(sql)

print("load_data.sql generated successfully!")

load_data.sql generated successfully!


In [ ]:
df.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
with open("load_data.sql", "r") as file:
    sql_content = file.read()

print("Total INSERT statements:",
      sql_content.count("INSERT INTO"))

print("\nClinic INSERT statements:",
      sql_content.count("INSERT INTO clinics"))

print("\nAppointment INSERT statements:",
      sql_content.count("INSERT INTO appointments"))

print("\nFirst 500 characters:")
print(sql_content[:500])

Total INSERT statements: 998

Clinic INSERT statements: 8

Appointment INSERT statements: 990

First 500 characters:

INSERT INTO clinics
(clinic_id, clinic_name, city, tier, latitude, longitude, temp_c)
VALUES (
    'C01',
    'Kothrud Family Clinic',
    'Pune',
    'Metro',
    18.5074,
    73.8077,
    18.3
);

INSERT INTO clinics
(clinic_id, clinic_name, city, tier, latitude, longitude, temp_c)
VALUES (
    'C02',
    'Hadapsar Care Centre',
    'Pune',
    'Metro',
    18.5089,
    73.926,
    20.0
);

INSERT INTO clinics
(clinic_id, clinic_name, city, tier, latitude, longitude, temp_c)
VALUES (
    'C03


In [ ]:
                     # Part 2

In [ ]:
# Task 2.2

In [ ]:
df.columns

Index(['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
       'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
       'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days',
       'no_show'],
      dtype='object')

In [ ]:
# Grouping appointments by clinic

clinic_counts = (
    df.groupby("clinic_id")
      .agg(
          appointments=("appointment_id", "count"),
          no_shows=("no_show", "sum")
      )
      .reset_index()
)

clinic_counts


,clinic_id,appointments,no_shows
0,C01,160,36
1,C02,143,39
2,C03,82,13
3,C04,142,38
4,C05,130,18
5,C06,142,23
6,C07,101,21
7,C08,90,30


In [ ]:
# Calculating the no-show percentage

clinic_counts["no_show_pct"] = (
    clinic_counts["no_shows"] /
    clinic_counts["appointments"] * 100
).round(1)

clinic_counts

,clinic_id,appointments,no_shows,no_show_pct
0,C01,160,36,22.5
1,C02,143,39,27.3
2,C03,82,13,15.9
3,C04,142,38,26.8
4,C05,130,18,13.8
5,C06,142,23,16.2
6,C07,101,21,20.8
7,C08,90,30,33.3


In [ ]:
# Adding clinic names and cities

clinic_counts = clinic_counts.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id"
)

clinic_counts

,clinic_id,appointments,no_shows,no_show_pct,clinic_name,city
0,C01,160,36,22.5,Kothrud Family Clinic,Pune
1,C02,143,39,27.3,Hadapsar Care Centre,Pune
2,C03,82,13,15.9,Sitabuldi Health Point,Nagpur
3,C04,142,38,26.8,Malviya Nagar Clinic,Jaipur
4,C05,130,18,13.8,Vaishali Nagar Clinic,Jaipur
5,C06,142,23,16.2,Edappally Medical Centre,Kochi
6,C07,101,21,20.8,Vijay Nagar Clinic,Indore
7,C08,90,30,33.3,Gomti Nagar Care Point,Lucknow


In [ ]:
# Sorting by no-show percentage

clinic_counts = clinic_counts.sort_values(
    by="no_show_pct",
    ascending=False
)

clinic_counts

,clinic_id,appointments,no_shows,no_show_pct,clinic_name,city
7,C08,90,30,33.3,Gomti Nagar Care Point,Lucknow
1,C02,143,39,27.3,Hadapsar Care Centre,Pune
3,C04,142,38,26.8,Malviya Nagar Clinic,Jaipur
0,C01,160,36,22.5,Kothrud Family Clinic,Pune
6,C07,101,21,20.8,Vijay Nagar Clinic,Indore
5,C06,142,23,16.2,Edappally Medical Centre,Kochi
2,C03,82,13,15.9,Sitabuldi Health Point,Nagpur
4,C05,130,18,13.8,Vaishali Nagar Clinic,Jaipur


In [ ]:
# Verifying the total counts

print("Total appointments:", clinic_counts["appointments"].sum())
print("Total no-shows:", clinic_counts["no_shows"].sum())

Total appointments: 990
Total no-shows: 218
